In [ ]:
from pathlib import Path
import numpy as np
from scipy.io import loadmat
from scipy import signal
from scipy.stats import kurtosis

registered_dir = Path('/Users/rustin/Documents/Big Data 567/SuperEeg-M467-project/registered_outputs_no_duplicates')
data_root      = Path('/Users/rustin/Documents/Big Data 567/faces_basic/data')
out_dir        = Path('/Users/rustin/Documents/Big Data 567/All_Locs_MNI_and_Ecog_Preprocessed')
out_dir.mkdir(parents=True, exist_ok=True)

srate            = 1000   # Hz
kurtosis_thresh  = 10     # drop channels with kurtosis >= 10 (SuperEEG paper default)
sos_notch = signal.butter(4, [59.5, 60.5], btype='bandstop', fs=srate, output='sos')


In [ ]:
def preprocess_subject(npy_file):
    pid  = npy_file.stem.split('_')[0]
    locs = np.load(npy_file).astype(np.float32, copy=False)

    mat_path = data_root / pid / f'{pid}_faceshouses.mat'
    if not mat_path.exists():
        print(f'[{pid}] missing ECoG file, skipped'); return

    mat  = loadmat(mat_path, squeeze_me=True)
    ecog = np.asarray(mat['data'], dtype=np.float32)
    if ecog.ndim != 2:
        print(f'[{pid}] unexpected ecog shape {ecog.shape}, skipped'); return

    # Align channels to de-duplicated locs if a keep_idx file exists
    if ecog.shape[1] != locs.shape[0]:
        keep_idx_path = registered_dir / f'{pid}_xslocs_registered_no_dups_keep_idx.npy'
        if keep_idx_path.exists():
            keep_idx = np.load(keep_idx_path).astype(int)
            ecog = ecog[:, keep_idx]
        else:
            print(f'[{pid}] channel/loc mismatch ({ecog.shape[1]} vs {locs.shape[0]}), skipped'); return

    # Keep only samples where stimulus is non-zero
    stim = mat.get('stim')
    if stim is not None:
        stim = np.asarray(stim).ravel()
        if stim.shape[0] == ecog.shape[0]:
            ecog = ecog[stim != 0]

    # 60 Hz notch filter
    ecog = signal.sosfiltfilt(sos_notch, ecog, axis=0).astype(np.float32, copy=False)

    # Drop high-kurtosis channels
    good = np.where(kurtosis(ecog, axis=0) <= kurtosis_thresh)[0]
    if good.size < 2:
        print(f'[{pid}] only {good.size} channels after kurtosis filter, skipped'); return

    np.savez_compressed(
        out_dir / f'{pid}_preprocessed.npz',
        subject_id=np.array(pid),
        sample_rate_hz=np.int32(srate),
        locs_mni_mm=locs[good],
        ecog=ecog[:, good],
    )
    print(f'[{pid}] kept {good.size}/{locs.shape[0]} channels  |  ecog={ecog[:, good].shape}')


In [ ]:
for npy_file in sorted(registered_dir.glob('*_xslocs_registered_no_dups.npy')):
    preprocess_subject(npy_file)

print('Done.')
